In [1]:
import numpy as np
import matplotlib.pyplot as plt
import emodeconnection as emc

from datetime import datetime

# Wavelength for the first calculation
wavelength_nm = 1546.5

# Materials
core_material = "Si"         # Silicon core
cladding_material = "SiO2"   # Silica cladding

# Rectangular core dimensions
core_width_nm = 500
core_height_nm = 220

# Distance from the core to the simulation boundaries
side_cladding_nm = 1500      # Left and right
vertical_cladding_nm = 1000  # Above and below

# Simulation window dimensions
window_width_nm = core_width_nm + 2 * side_cladding_nm
window_height_nm = core_height_nm + 2 * vertical_cladding_nm

# Simulation mesh step
mesh_nm = 5

# Request multiple solutions to inspect the modes
num_modes = 4

# Unique name for each run
simulation_name = (
    f"Si_waveguide_{wavelength_nm:g}_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)

print(f"Simulation: {simulation_name}")
print(f"Core: {core_width_nm} × {core_height_nm} nm")
print(f"Window: {window_width_nm} × {window_height_nm} nm")
print(f"Wavelength: {wavelength_nm} nm")
print(f"Mesh: {mesh_nm} nm")

Simulation: Si_waveguide_1546.5_20261030_130442_779263
Core: 500 × 220 nm
Window: 3500 × 2220 nm
Wavelength: 1546.5 nm
Mesh: 5 nm


In [ ]:
em = None

try:
    # Open an EMode session
    em = emc.EMode(
        simulation_name=simulation_name,
        license_type="2d",
        clear="mine",
        verbose=True,
    )

    # Simulation window, mesh, and background material
    em.settings(
        wavelength=wavelength_nm,
        x_resolution=mesh_nm,
        y_resolution=mesh_nm,
        window_width=window_width_nm,
        window_height=window_height_nm,
        background_material=cladding_material,
        boundary_condition="00",
        num_modes=num_modes,
    )

    # Rectangular core centered in the simulation window
    em.shape(
        name="core",
        material=core_material,
        shape_type="planar",
        width=core_width_nm,
        height=core_height_nm,
        position=[0, window_height_nm / 2],
    )

    # Calculate the cross-sectional modes
    em.FDM()

    # Get effective indices and TE polarization fractions
    neff = np.asarray(em.get("effective_index")).reshape(-1)
    te_fraction = np.asarray(em.get("TE_fraction")).reshape(-1)

    print("\nCalculated modes:")
    for i, (n, te) in enumerate(zip(neff, te_fraction)):
        print(
            f"Mode {i}: n_eff = {n:.9f}, "
            f"TE fraction = {te:.4f}"
        )

    # Candidate fundamental TE mode:
    # highest n_eff among predominantly TE solutions
    te_candidates = np.flatnonzero(te_fraction > 0.5)

    if te_candidates.size == 0:
        raise RuntimeError(
            "No predominantly TE modes found. Check the mode table."
        )

    selected_mode = int(
        te_candidates[np.argmax(np.real(neff[te_candidates]))]
    )

    print("\nSelected mode:", selected_mode)
    print("Selected n_eff:", neff[selected_mode])

    # Refractive-index map
    em.plot(
        component="Index",
        aspect_ratio="real",
        file_name=simulation_name + "_index",
        file_type="png",
    )

    # Both transverse components of the selected mode
    for component in ["Ex", "Ey"]:
        em.plot(
            component=component,
            mode=selected_mode,
            plot_function="abs",
            aspect_ratio="real",
            index_outline=True,
            file_name=simulation_name + "_" + component,
            file_type="png",
        )

finally:
    # Close the session even if the calculation raises an error
    if em is not None:
        em.close()
        print("EMode session closed.")

In [ ]:
# Results from individual EMode runs:
# 5 nm mesh, 3500 × 2220 nm window
wavelengths_nm = np.array([1547, 1549, 1550, 1551, 1553])

neff_values = np.array([
    2.4505315510144174,
    2.448275829459585,
    2.447147752683864,
    2.4460195657811297,
    2.4437630980461074,
])

plt.figure(figsize=(7, 4))
plt.plot(wavelengths_nm, neff_values, "o-")
plt.xlabel("Wavelength (nm)")
plt.ylabel("Effective index")
plt.title("Fundamental TE mode")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Use values from the table in the previous cell
n_1549 = neff_values[wavelengths_nm == 1549].item()
n_1550 = neff_values[wavelengths_nm == 1550].item()
n_1551 = neff_values[wavelengths_nm == 1551].item()

# Slope near 1550 nm: central difference
slope_per_nm = (n_1551 - n_1549) / (1551.0 - 1549.0)

# Group index at 1550 nm
ng_1550 = n_1550 - 1550.0 * slope_per_nm

print(f"dn_eff/dλ = {slope_per_nm:.9f} 1/nm")
print(f"n_eff     = {n_1550:.6f}")
print(f"n_g       = {ng_1550:.6f}")

In [ ]:
# Initial AWG model parameters
lambda_center_nm = 1550.0
n_channels = 4
channel_spacing_nm = 2.0
fsr_target_nm = n_channels * channel_spacing_nm

# The AWG order is an integer
order_estimate = (
    lambda_center_nm * n_1550 / (ng_1550 * fsr_target_nm)
)
awg_order = int(round(order_estimate))

# Length difference between adjacent arms
delta_L_nm = awg_order * lambda_center_nm / n_1550

# FSR after selecting an integer order (local approximation)
fsr_estimate_nm = (
    lambda_center_nm**2 / (ng_1550 * delta_L_nm)
)

print(f"Order before rounding: {order_estimate:.6f}")
print(f"Selected order m:      {awg_order}")
print(f"Length step ΔL:        {delta_L_nm / 1000:.6f} µm")
print(f"Estimated FSR:         {fsr_estimate_nm:.6f} nm")

In [ ]:
# Number of phase cycles over the additional length ΔL
phase_turns = neff_values * delta_L_nm / wavelengths_nm

# Subtract 113 full cycles: the reference value at 1550 nm
residual_turns = phase_turns - awg_order
residual_phase_deg = residual_turns * 360

for wl, turns, phase_deg in zip(
    wavelengths_nm, phase_turns, residual_phase_deg
):
    print(
        f"{wl} nm: {turns:.6f} turns, "
        f"residual phase = {phase_deg:+.2f} deg"
    )

In [ ]:
path_indices = np.arange(8)

plt.figure(figsize=(8, 4))

for wl, phase_step in zip(wavelengths_nm, residual_phase_deg):
    # Phase of arm j relative to arm 0
    phase_along_array = path_indices * phase_step

    plt.plot(
        path_indices,
        phase_along_array,
        "o-",
        label=f"{wl} nm",
    )

plt.xlabel("Path index j")
plt.ylabel("Relative phase, full 113-turn steps removed (deg)")
plt.title("Phase progression across the array")
plt.xticks(path_indices)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Four operating channels; 1550 nm is only the reference point
channel_mask = wavelengths_nm != 1550
channels_nm = wavelengths_nm[channel_mask]

# Phase steps between adjacent arms, in radians
channel_phase_steps = np.deg2rad(
    residual_phase_deg[channel_mask]
)

number_of_paths = 8
j = np.arange(number_of_paths)

# Rows: output ports; columns: input wavelengths
response = np.zeros((4, 4))

for port in range(4):
    # The output is matched to the phase step of its channel
    port_phase_step = channel_phase_steps[port]

    for channel in range(4):
        phase_mismatch = (
            channel_phase_steps[channel] - port_phase_step
        )

        # Sum complex fields, then calculate |A|²
        amplitude = np.mean(np.exp(1j * j * phase_mismatch))
        response[port, channel] = np.abs(amplitude)**2

print("Columns: wavelengths (nm):", channels_nm)
print("Rows: ports 0, 1, 2, 3")
print(np.array2string(response, precision=6, suppress_small=True))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(response, vmin=0, vmax=1, cmap="viridis")

ax.set_xticks(range(4), labels=channels_nm)
ax.set_yticks(
    range(4),
    labels=[f"Port {p}: {wl} nm" for p, wl in enumerate(channels_nm)]
)
ax.set_xlabel("Input wavelength (nm)")
ax.set_ylabel("Output port")
ax.set_title("Ideal array: normalized response")

for port in range(4):
    for channel in range(4):
        value = response[port, channel]
        ax.text(
            channel, port, f"{value:.4f}",
            ha="center", va="center",
            color="black" if value > 0.5 else "white"
        )

fig.colorbar(im, ax=ax, label="Normalized response")
plt.tight_layout()
plt.show()

In [ ]:
# Restrict the range to the available EMode data
spectrum_wavelengths_nm = np.linspace(1547, 1553, 1201)

# Linear interpolation between simulated values
spectrum_neff = np.interp(
    spectrum_wavelengths_nm,
    wavelengths_nm,
    neff_values,
)

# Phase step between adjacent arms
spectrum_phase_steps = 2 * np.pi * (
    spectrum_neff * delta_L_nm / spectrum_wavelengths_nm
    - awg_order
)

# Keep the previous model: 8 arms with equal amplitudes
number_of_paths = 24
j = np.arange(number_of_paths)

spectra = np.zeros((len(spectrum_wavelengths_nm), len(channels_nm)))

for port, port_phase_step in enumerate(channel_phase_steps):
    phase_mismatch = spectrum_phase_steps - port_phase_step

    # Sum the fields from all arms at each wavelength
    fields = np.exp(1j * phase_mismatch[:, None] * j[None, :])
    amplitude = np.mean(fields, axis=1)

    spectra[:, port] = np.abs(amplitude)**2

# Two views of the same result
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for port, wl in enumerate(channels_nm):
    label = f"Port {port}: {wl} nm"

    axes[0].plot(spectrum_wavelengths_nm, spectra[:, port],
                 label=label)

    spectra_db = 10 * np.log10(
        np.maximum(spectra[:, port], 1e-12)
    )
    axes[1].plot(spectrum_wavelengths_nm, spectra_db,
                 label=label)

axes[0].set_ylabel("Normalized response")
axes[0].set_ylim(0, 1.05)

axes[1].set_ylabel("Normalized response (dB)")
axes[1].set_ylim(-50, 1)

for ax in axes:
    ax.set_xlabel("Wavelength (nm)")
    ax.grid(True)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
number_of_paths = 24
j = np.arange(number_of_paths)

# Distance from the array center: from -1 to +1
position = (j - (number_of_paths - 1) / 2) / (
    (number_of_paths - 1) / 2
)

# Field amplitudes: outermost arms have 15% of the central value
edge_amplitude = 0.15
weights = np.exp(np.log(edge_amplitude) * position**2)

# Select the output for 1549 nm
port = np.argmin(np.abs(channels_nm - 1549))
mismatch = spectrum_phase_steps - channel_phase_steps[port]

fields = np.exp(1j * mismatch[:, None] * j[None, :])

# Uniform and Gaussian amplitude distributions
amplitude_uniform = np.mean(fields, axis=1)
amplitude_gaussian = (fields @ weights) / np.sum(weights)

response_uniform = np.abs(amplitude_uniform)**2
response_gaussian = np.abs(amplitude_gaussian)**2

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(j, np.ones(number_of_paths), "o--", label="Uniform")
axes[0].plot(j, weights, "o-", label="Gaussian")
axes[0].set_xlabel("Arm index")
axes[0].set_ylabel("Relative field amplitude")
axes[0].set_title("Contribution of each arm")
axes[0].grid(True)
axes[0].legend()

for response, label in [
    (response_uniform, "Uniform"),
    (response_gaussian, "Gaussian"),
]:
    axes[1].plot(
        spectrum_wavelengths_nm,
        10 * np.log10(np.maximum(response, 1e-12)),
        label=label,
    )

axes[1].set_xlabel("Wavelength, nm")
axes[1].set_ylabel("Normalized response, dB")
axes[1].set_title(f"Output for {channels_nm[port]:g} nm")
axes[1].set_ylim(-60, 1)
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
number_of_paths = 24
j = np.arange(number_of_paths)

position = (j - (number_of_paths - 1) / 2) / (
    (number_of_paths - 1) / 2
)

weights_uniform = np.ones(number_of_paths)
weights_gaussian = np.exp(np.log(0.15) * position**2)

# Rows: outputs. Columns: input wavelengths.
mismatch = (
    channel_phase_steps[None, :]
    - channel_phase_steps[:, None]
)

fields = np.exp(1j * mismatch[:, :, None] * j)

off_diagonal = ~np.eye(len(channels_nm), dtype=bool)

for name, weights in [
    ("Uniform weights", weights_uniform),
    ("Gaussian weights", weights_gaussian),
]:
    amplitude = np.sum(fields * weights, axis=2) / np.sum(weights)
    power = np.abs(amplitude)**2

    # Relative to the desired signal at each output
    relative_power = power / np.diag(power)[:, None]
    response_db = 10 * np.log10(np.maximum(relative_power, 1e-12))

    print(f"\n{name}")
    print("Output ↓ / Input →", end="")
    for wavelength in channels_nm:
        print(f"{wavelength:>10.0f}", end="")
    print()

    for wavelength, row in zip(channels_nm, response_db):
        print(f"{wavelength:>15.0f}", end="")
        for value in row:
            print(f"{value:>10.2f}", end="")
        print()

    worst_crosstalk_db = np.max(response_db[off_diagonal])
    print(f"Worst-case crosstalk: {worst_crosstalk_db:.2f} dB")

In [ ]:
test_wavelengths_nm = np.array([1548.9, 1549.0, 1549.1])

# All test points are within the simulated range
test_neff = np.interp(
    test_wavelengths_nm, wavelengths_nm, neff_values
)

test_phase_steps = 2 * np.pi * (
    test_neff * delta_L_nm / test_wavelengths_nm - awg_order
)

target_port = np.argmin(np.abs(channels_nm - 1549))
other_ports = np.arange(len(channels_nm)) != target_port

j = np.arange(24)
position = (j - (len(j) - 1) / 2) / ((len(j) - 1) / 2)

for name, weights in [
    ("Uniform weights", np.ones(len(j))),
    ("Gaussian weights", np.exp(np.log(0.15) * position**2)),
]:
    print(f"\n{name}")
    print("λ, nm     Desired response, dB     Leakage / desired, dB")

    for wavelength, phase_step in zip(
        test_wavelengths_nm, test_phase_steps
    ):
        # Keep the settings of all four outputs unchanged
        mismatch = phase_step - channel_phase_steps
        fields = np.exp(1j * mismatch[:, None] * j[None, :])

        amplitude = (fields @ weights) / np.sum(weights)
        power = np.abs(amplitude)**2

        useful_power = power[target_port]
        largest_leakage = np.max(power[other_ports])

        useful_db = 10 * np.log10(max(useful_power, 1e-12))
        leakage_db = 10 * np.log10(
            max(largest_leakage / useful_power, 1e-12)
        )

        print(
            f"{wavelength:7.1f}"
            f"{useful_db:24.2f}"
            f"{leakage_db:27.2f}"
        )

In [ ]:
offsets_nm = np.linspace(-0.3, 0.3, 1201)
test_wavelengths_nm = 1549.0 + offsets_nm

test_neff = np.interp(
    test_wavelengths_nm, wavelengths_nm, neff_values
)
test_phase_steps = 2 * np.pi * (
    test_neff * delta_L_nm / test_wavelengths_nm - awg_order
)

j = np.arange(24)
position = (j - (len(j) - 1) / 2) / ((len(j) - 1) / 2)

target_port = np.argmin(np.abs(channels_nm - 1549))
other_ports = np.arange(len(channels_nm)) != target_port

# Dimensions: wavelength × output × arm
mismatch = test_phase_steps[:, None] - channel_phase_steps[None, :]
fields = np.exp(1j * mismatch[:, :, None] * j)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, weights in [
    ("Uniform", np.ones(len(j))),
    ("Gaussian", np.exp(np.log(0.15) * position**2)),
]:
    amplitude = (fields @ weights) / np.sum(weights)
    power = np.abs(amplitude)**2

    useful = power[:, target_port]
    leakage = np.max(power[:, other_ports], axis=1)

    useful_db = 10 * np.log10(np.maximum(useful, 1e-12))
    leakage_db = 10 * np.log10(
        np.maximum(leakage / np.maximum(useful, 1e-12), 1e-12)
    )

    acceptable = (useful_db >= -1) & (leakage_db <= -30)

    # Find the contiguous valid interval around zero offset
    center = np.argmin(np.abs(offsets_nm))
    left = right = center

    if acceptable[center]:
        while left > 0 and acceptable[left - 1]:
            left -= 1
        while right < len(offsets_nm) - 1 and acceptable[right + 1]:
            right += 1

        symmetric_tolerance = min(-offsets_nm[left], offsets_nm[right])

        print(
            f"{name}: from {offsets_nm[left]:+.4f} "
            f"to {offsets_nm[right]:+.4f} nm"
        )
        print(f"  Symmetric tolerance: ±{symmetric_tolerance:.4f} nm")

        if left == 0 or right == len(offsets_nm) - 1:
            print("  The interval reached the sweep boundary.")
    else:
        print(f"{name}: criteria not met even at the center.")

    axes[0].plot(offsets_nm, useful_db, label=name)
    axes[1].plot(offsets_nm, leakage_db, label=name)

axes[0].axhline(-1, color="black", linestyle="--", label="−1 dB threshold")
axes[0].set_ylabel("Desired response, dB")
axes[0].set_ylim(-6, 0.2)

axes[1].axhline(-30, color="black", linestyle="--", label="−30 dB threshold")
axes[1].set_ylabel("Leakage / desired signal, dB")

for ax in axes:
    ax.set_xlabel("Offset from 1549 nm, nm")
    ax.grid(True)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Extended EMode data
wl_extended = np.array([
    1546.5, 1547.0, 1549.0, 1550.0, 1551.0, 1553.0, 1553.3
])

neff_extended = np.array([
    2.451095554,
    2.4505315510144174,
    2.448275829459585,
    2.447147752683864,
    2.4460195657811297,
    2.4437630980461074,
    2.443424560,
])

# Existing AWG design
channels_check = np.array([1547.0, 1549.0, 1551.0, 1553.0])
order_check = 113
delta_L_check = order_check * 1550.0 / neff_extended[3]

# Fixed output settings
port_phases = 2 * np.pi * (
    np.interp(channels_check, wl_extended, neff_extended)
    * delta_L_check / channels_check - order_check
)

j = np.arange(24)
position = (j - 11.5) / 11.5
offsets = np.linspace(-0.3, 0.3, 1201)
center = len(offsets) // 2

for name, weights in [
    ("Uniform", np.ones(24)),
    ("Gaussian", np.exp(np.log(0.15) * position**2)),
]:
    print(f"\n{name} weights")
    print("Channel, nm    Lower Δλ    Upper Δλ    Tolerance ±, nm")
    tolerances = []

    for port, wavelength in enumerate(channels_check):
        test_wl = wavelength + offsets

        # Do not evaluate outside the EMode data range
        assert test_wl.min() >= wl_extended.min() - 1e-9
        assert test_wl.max() <= wl_extended.max() + 1e-9

        test_neff = np.interp(test_wl, wl_extended, neff_extended)
        phase = 2 * np.pi * (
            test_neff * delta_L_check / test_wl - order_check
        )

        mismatch = phase[:, None] - port_phases[None, :]
        fields = np.exp(1j * mismatch[:, :, None] * j)
        amplitude = (fields @ weights) / weights.sum()
        power = np.abs(amplitude)**2

        useful = power[:, port]
        other_ports = np.arange(4) != port
        leakage = np.max(power[:, other_ports], axis=1)

        useful_db = 10 * np.log10(np.maximum(useful, 1e-12))
        leakage_db = 10 * np.log10(
            np.maximum(leakage / np.maximum(useful, 1e-12), 1e-12)
        )

        acceptable = (useful_db >= -1) & (leakage_db <= -30)

        if not acceptable[center]:
            print(f"{wavelength:8.0f}    Criteria not met at the center")
            tolerances.append(0.0)
            continue

        left = right = center
        while left > 0 and acceptable[left - 1]:
            left -= 1
        while right < len(offsets) - 1 and acceptable[right + 1]:
            right += 1

        tolerance = min(-offsets[left], offsets[right])
        tolerances.append(tolerance)

        print(
            f"{wavelength:8.0f}"
            f"{offsets[left]:14.4f}"
            f"{offsets[right]:14.4f}"
            f"{tolerance:16.4f}"
        )

        if left == 0 or right == len(offsets) - 1:
            print("  Sweep boundary reached.")

    print(
        f"Common symmetric tolerance for all channels: "
        f"±{min(tolerances):.4f} nm"
    )

In [ ]:
# Reproducible set of phase errors for 24 arms
rng = np.random.default_rng(42)
sigma_phase_rad = 0.05

phase_errors = rng.normal(
    loc=0.0, scale=sigma_phase_rad, size=len(j)
)

# Remove the common phase: it does not affect power
phase_errors -= phase_errors.mean()

print(
    "RMS error in this realization: "
    f"{np.sqrt(np.mean(phase_errors**2)):.4f} rad"
)

# Rows: outputs. Columns: input channels.
mismatch_centers = (
    port_phases[None, :] - port_phases[:, None]
)

for name, weights in [
    ("Uniform", np.ones(len(j))),
    ("Gaussian", np.exp(np.log(0.15) * position**2)),
]:
    print(f"\n{name} weights")
    print("Case       Desired response, dB    Worst leakage, dB")

    for case, errors in [
        ("Without errors", np.zeros(len(j))),
        ("With errors", phase_errors),
    ]:
        fields = np.exp(
            1j * (
                mismatch_centers[:, :, None] * j
                + errors[None, None, :]
            )
        )

        amplitude = (fields @ weights) / weights.sum()
        power = np.abs(amplitude)**2

        # Diagonal: power at the desired output for each laser
        useful = np.diag(power)
        useful_db = 10 * np.log10(np.maximum(useful, 1e-12))

        # Normalize each column by the desired output for that laser
        relative_power = power / useful[None, :]
        off_diagonal = ~np.eye(len(channels_check), dtype=bool)

        worst_leakage_db = 10 * np.log10(
            max(np.max(relative_power[off_diagonal]), 1e-12)
        )

        print(
            f"{case:<13}"
            f"{np.min(useful_db):>20.2f}"
            f"{worst_leakage_db:>22.2f}"
        )

In [ ]:
n_trials = 1000
sigma_phase_rad = 0.05
rng = np.random.default_rng(42)

j_mc = np.arange(24)
position_mc = (j_mc - 11.5) / 11.5

# Independent arm errors; each realization is shared by all four channels
errors_mc = rng.normal(
    0.0, sigma_phase_rad, size=(n_trials, len(j_mc))
)
errors_mc -= errors_mc.mean(axis=1, keepdims=True)

# Fixed output settings from the previous cell
mismatch_mc = port_phases[None, :] - port_phases[:, None]

# Dimensions: trial × output × input channel × arm
fields_mc = np.exp(
    1j * (
        mismatch_mc[None, :, :, None] * j_mc
        + errors_mc[:, None, None, :]
    )
)

off_diagonal = ~np.eye(len(port_phases), dtype=bool)
mc_results = {}

for name, weights in [
    ("Uniform", np.ones(24)),
    ("Gaussian", np.exp(np.log(0.15) * position_mc**2)),
]:
    amplitude = (fields_mc @ weights) / weights.sum()
    power = np.abs(amplitude)**2

    # Desired power: trial × channel
    useful = np.diagonal(power, axis1=1, axis2=2)
    useful_db = 10 * np.log10(np.maximum(useful, 1e-12))
    worst_useful_db = useful_db.min(axis=1)

    # For each laser: undesired output / desired output
    relative_power = power / np.maximum(useful[:, None, :], 1e-12)

    # Largest individual leakage across all channel pairs
    worst_leakage = relative_power[:, off_diagonal].max(axis=1)
    worst_leakage_db = 10 * np.log10(
        np.maximum(worst_leakage, 1e-12)
    )

    passed = (
        (worst_useful_db >= -1)
        & (worst_leakage_db <= -30)
    )

    mc_results[name] = {
        "useful_db": worst_useful_db,
        "leakage_db": worst_leakage_db,
        "passed": passed,
    }

    print(f"\n{name} weights")
    print(
        "Median worst-case leakage: "
        f"{np.median(worst_leakage_db):.2f} dB"
    )
    print(
        "95-th percentile of worst-case leakage: "
        f"{np.percentile(worst_leakage_db, 95):.2f} dB"
    )
    print(
        "Maximum leakage across 1000 trials: "
        f"{worst_leakage_db.max():.2f} dB"
    )
    print(
        "Minimum desired response across all trials: "
        f"{worst_useful_db.min():.3f} dB"
    )
    print(
        f"Passed both criteria: {passed.sum()}/{n_trials} "
        f"({100 * passed.mean():.1f}%)"
    )

In [ ]:
import matplotlib.pyplot as plt

offsets_combined = np.linspace(-0.15, 0.15, 121)
off_diagonal = ~np.eye(len(channels_check), dtype=bool)

weight_options = {
    "Uniform": np.ones(len(j_mc)),
    "Gaussian": np.exp(np.log(0.15) * position_mc**2),
}

# Store the result of each trial at every offset
combined_pass = {
    name: np.zeros(
        (len(offsets_combined), len(errors_mc)), dtype=bool
    )
    for name in weight_options
}

for k, offset in enumerate(offsets_combined):
    test_wl = channels_check + offset

    assert test_wl.min() >= wl_extended.min()
    assert test_wl.max() <= wl_extended.max()

    test_neff = np.interp(test_wl, wl_extended, neff_extended)
    input_phases = 2 * np.pi * (
        test_neff * delta_L_check / test_wl - order_check
    )

    # Rows: outputs; columns: shifted input channels
    mismatch = input_phases[None, :] - port_phases[:, None]

    fields = np.exp(
        1j * (
            mismatch[None, :, :, None] * j_mc
            + errors_mc[:, None, None, :]
        )
    )

    for name, weights in weight_options.items():
        amplitude = (fields @ weights) / weights.sum()
        power = np.abs(amplitude)**2

        useful = np.diagonal(power, axis1=1, axis2=2)
        useful_db = 10 * np.log10(np.maximum(useful, 1e-12))

        relative_power = power / np.maximum(
            useful[:, None, :], 1e-12
        )
        worst_leakage = relative_power[:, off_diagonal].max(axis=1)
        worst_leakage_db = 10 * np.log10(
            np.maximum(worst_leakage, 1e-12)
        )

        combined_pass[name][k] = (
            (useful_db.min(axis=1) >= -1)
            & (worst_leakage_db <= -30)
        )

plt.figure(figsize=(8, 4.5))

for name, passed in combined_pass.items():
    pass_percent = 100 * passed.mean(axis=1)
    plt.plot(offsets_combined, pass_percent, label=name)

plt.axhline(95, color="black", linestyle="--", label="95% level")
plt.xlabel("Offset from the nominal wavelength, nm")
plt.ylabel("Trials passing both criteria, %")
plt.title("Phase errors σ = 0.05 rad + laser wavelength offsets")
plt.ylim(0, 101)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

print("Offset, nm    Uniform, %    Gaussian, %")
for offset in [-0.10, -0.05, 0.0, 0.05, 0.10]:
    k = np.argmin(np.abs(offsets_combined - offset))
    uniform = 100 * combined_pass["Uniform"][k].mean()
    gaussian = 100 * combined_pass["Gaussian"][k].mean()
    print(f"{offset:+12.3f}{uniform:18.1f}{gaussian:15.1f}")

In [ ]:
# Candidate half-widths of the symmetric interval [-radius, +radius]
radii = np.maximum(
    offsets_combined[offsets_combined >= -1e-12], 0.0
)

for name, passed in combined_pass.items():
    interval_yield = []

    for radius in radii:
        mask = np.abs(offsets_combined) <= radius + 1e-12

        # A trial passes only if the criteria are met
        # at ALL tested offsets within the interval.
        passed_entire_interval = passed[mask].all(axis=0)
        interval_yield.append(passed_entire_interval.mean())

    interval_yield = np.array(interval_yield)
    acceptable = np.flatnonzero(interval_yield >= 0.95)

    print(f"\n{name}")

    if acceptable.size:
        k = acceptable[-1]
        print(f"Tolerance: ±{radii[k]:.4f} nm")
        print(
            f"Passed the entire interval: "
            f"{100 * interval_yield[k]:.1f}% trials"
        )
    else:
        print("Fewer than 95% of trials pass even at zero offset.")

In [ ]:
sigma_values = [0.00, 0.02, 0.04, 0.05, 0.06, 0.08, 0.10]
n_trials_sweep = 1000

# Use the same random deviations for all values of σ
rng_sweep = np.random.default_rng(42)
base_errors = rng_sweep.normal(
    0.0, 1.0, size=(n_trials_sweep, len(j_mc))
)
base_errors -= base_errors.mean(axis=1, keepdims=True)

# Offset grid: 0.0025 nm step
offsets_sweep = np.linspace(-0.15, 0.15, 121)
center = len(offsets_sweep) // 2

# Precompute the response without random phase errors
test_wl = channels_check[None, :] + offsets_sweep[:, None]

assert test_wl.min() >= wl_extended.min()
assert test_wl.max() <= wl_extended.max()

test_neff = np.interp(test_wl, wl_extended, neff_extended)
input_phases = 2 * np.pi * (
    test_neff * delta_L_check / test_wl - order_check
)

# Dimensions: offset × output × input × arm
mismatch = input_phases[:, None, :] - port_phases[None, :, None]
ideal_fields = np.exp(1j * mismatch[..., None] * j_mc)

n_channels = len(channels_check)
off_diagonal_sweep = ~np.eye(n_channels, dtype=bool)
sweep_results = {}

for sigma in sigma_values:
    phase_errors = np.exp(1j * sigma * base_errors)
    sweep_results[sigma] = {}

    for name, weights in weight_options.items():
        # Dimensions: error realization × offset × output × input
        amplitude = np.einsum(
            "soij,tj->tsoi",
            ideal_fields,
            phase_errors * weights[None, :],
            optimize=True,
        ) / weights.sum()

        power = np.abs(amplitude)**2
        useful = np.diagonal(power, axis1=2, axis2=3)

        useful_db = 10 * np.log10(np.maximum(useful, 1e-15))
        relative_power = power / np.maximum(
            useful[:, :, None, :], 1e-15
        )
        worst_leakage_db = 10 * np.log10(
            np.maximum(
                relative_power[:, :, off_diagonal_sweep].max(axis=-1),
                1e-15,
            )
        )

        passed = (
            (useful_db.min(axis=-1) >= -1.0)
            & (worst_leakage_db <= -30.0)
        )

        # Expand the interval from the center.
        # Each trial must pass all included offsets.
        survives = passed[:, center].copy()
        best_radius = None

        for step in range(center + 1):
            survives &= passed[:, center - step]
            survives &= passed[:, center + step]

            if survives.mean() >= 0.95:
                best_radius = float(
                    offsets_sweep[center + step]
                )
            else:
                break

        sweep_results[sigma][name] = best_radius

    print(f"Done: σ = {sigma:.2f} rad")

print("\nSymmetric tolerance with ≥95% of trials passing")
print("σ, rad    Uniform, nm    Gaussian, nm")

def format_radius(value):
    return "none" if value is None else f"±{value:.4f}"

for sigma, result in sweep_results.items():
    uniform = format_radius(result["Uniform"])
    gaussian = format_radius(result["Gaussian"])
    print(f"{sigma:6.2f}    {uniform:>15}    {gaussian:>12}")

In [ ]:
def interval_tolerance(passed):
    """Tolerance met by ≥95% of trials throughout the interval."""
    survives = passed[:, center].copy()
    best_radius = None

    for step in range(center + 1):
        survives &= passed[:, center - step]
        survives &= passed[:, center + step]

        if survives.mean() < 0.95:
            break

        best_radius = float(offsets_sweep[center + step])

    return best_radius


print("Tolerances for individual criteria and for both criteria together")
print("σ, rad  Weights        Response ≥−1 dB   Leakage ≤−30 dB   Both criteria")

for sigma in [0.04, 0.05, 0.06]:
    phase_errors = np.exp(1j * sigma * base_errors)

    for name, weights in weight_options.items():
        amplitude = np.einsum(
            "soij,tj->tsoi",
            ideal_fields,
            phase_errors * weights[None, :],
            optimize=True,
        ) / weights.sum()

        power = np.abs(amplitude)**2
        useful = np.diagonal(power, axis1=2, axis2=3)

        useful_db = 10 * np.log10(np.maximum(useful, 1e-15))

        relative_power = power / np.maximum(
            useful[:, :, None, :], 1e-15
        )
        worst_leakage_db = 10 * np.log10(
            np.maximum(
                relative_power[:, :, off_diagonal_sweep].max(axis=-1),
                1e-15,
            )
        )

        # Check the criteria separately
        pass_useful = useful_db.min(axis=-1) >= -1.0
        pass_leakage = worst_leakage_db <= -30.0

        tolerance_useful = interval_tolerance(pass_useful)
        tolerance_leakage = interval_tolerance(pass_leakage)
        tolerance_both = interval_tolerance(
            pass_useful & pass_leakage
        )

        print(
            f"{sigma:6.2f}  {name:<12}"
            f"  {format_radius(tolerance_useful):>13}"
            f"  {format_radius(tolerance_leakage):>14}"
            f"  {format_radius(tolerance_both):>12}"
        )

# 1×4 AWG: comparison of uniform and Gaussian weights

## Objective

Compare the robustness of a four-channel AWG to wavelength offsets and random phase errors in the array arms for two amplitude distributions: uniform and Gaussian.

## Model parameters

| Parameter | Value |
|---|---|
| Core / cladding | Si / SiO₂ |
| Core cross-section | 500 × 220 nm |
| Mode | Fundamental TE mode |
| Reference wavelength | 1550 nm |
| Effective index at 1550 nm | ≈2.44715 |
| Group index near 1550 nm | ≈4.19575 |
| Channel center wavelengths | 1547, 1549, 1551, 1553 nm |
| Number of array arms | 24 |
| Length difference between adjacent arms | ≈71.573 µm |
| Interference order | 113 |
| Estimated FSR near 1550 nm | ≈8 nm |

Effective-index dispersion is included by linearly interpolating the mode-solver results.

## Amplitude distributions and normalization

Uniform weights:

$$
w_j = 1.
$$

Gaussian weights:

$$
w_j =
\exp\left[
\ln(0.15)
\left(\frac{j-11.5}{11.5}\right)^2
\right],
\qquad j=0,\ldots,23.
$$

These are field-amplitude weights, not power weights. The outermost arms have a weight of 0.15.

The output field is calculated by summing the contributions from all arms:

$$
A_p(\lambda)=
\frac{
\sum_{j=0}^{23}w_j
\exp\left\{i\left[j\bigl(\phi(\lambda)-\phi_p\bigr)
+\varepsilon_j\right]\right\}
}{
\sum_{j=0}^{23}w_j
},
$$

where

$$
\phi(\lambda)=
2\pi\left(\frac{n_\mathrm{eff}(\lambda)\Delta L}{\lambda}-m\right),
\qquad
\phi_p=\phi(\lambda_p).
$$

The normalized response is $P_p=|A_p|^2$.
For each weight distribution, the ideal peak is normalized to 1 (0 dB).

## Robustness evaluation

For each value of σ, 1000 random phase-error realizations are used. The initially sampled errors of individual arms follow a normal distribution with zero mean and standard deviation σ.

The mean error is subtracted from each realization: a common phase shift of all arms does not affect the calculated power.

The same error realizations are used to compare the weight distributions. When σ changes, the same random deviations are scaled. The errors of each realization remain fixed during the wavelength sweep.

Wavelength offsets are swept from −0.15 to +0.15 nm in steps of 0.0025 nm. At each step, the same offset is added to all four nominal wavelengths. The response to each input wavelength is calculated separately.

Pass criteria:

1. The desired response of every channel is ≥−1 dB relative to the ideal peak.
2. The power at any undesired output is ≤−30 dB relative to the power at the desired output for the same input wavelength.

The symmetric tolerance ±δ is the largest tested interval over which at least 95% of realizations meet both criteria at every tested offset for all four channels.

The −1 dB, −30 dB, and 95% thresholds are the selected criteria for this study.

## Results: combined tolerance

| σ, rad | Uniform weights, nm | Gaussian weights, nm |
|---|---:|---:|
| 0.00 | ±0.0500 | ±0.1150 |
| 0.02 | ±0.0375 | ±0.1125 |
| 0.04 | ±0.0250 | ±0.1100 |
| 0.05 | ±0.0175 | ±0.0675 |
| 0.06 | ±0.0050 | ±0.0225 |
| 0.08 | None | None |
| 0.10 | None | None |

“None” means that fewer than 95% of realizations pass even at zero offset; it does not mean that the desired signal is absent.

At σ = 0.05 rad, Gaussian weights increase the symmetric tolerance by approximately a factor of 3.86. For both distributions, 951 out of 1000 realizations pass throughout their respective tolerance intervals.

## Limiting criterion

| σ, rad | Weights | Response only, nm | Leakage only, nm | Both criteria, nm |
|---|---|---:|---:|---:|
| 0.04 | Uniform | ±0.0825 | ±0.0250 | ±0.0250 |
| 0.04 | Gaussian | ±0.1100 | ±0.1150 | ±0.1100 |
| 0.05 | Uniform | ±0.0825 | ±0.0175 | ±0.0175 |
| 0.05 | Gaussian | ±0.1100 | ±0.0675 | ±0.0675 |
| 0.06 | Uniform | ±0.0800 | ±0.0050 | ±0.0050 |
| 0.06 | Gaussian | ±0.1075 | ±0.0225 | ±0.0225 |

For uniform weights, leakage into undesired outputs limits the tolerance in all three cases.

For Gaussian weights, the desired response limits the tolerance at σ = 0.04 rad. At σ = 0.05 and 0.06 rad, crosstalk becomes the limiting criterion.

## Conclusion

In this model, the Gaussian amplitude distribution improves robustness to wavelength offsets, including in the presence of phase errors. However, it does not eliminate phase-error sensitivity: larger phase errors increase leakage into undesired outputs.

The sharp reduction in Gaussian tolerance between σ = 0.04 and 0.05 rad is associated with a change in the limiting criterion: the desired response still meets its threshold, but leakage exceeds the allowed level for some realizations.

## Scope and limitations

- The model sums interfering fields; it is not a full electromagnetic model of an AWG with star couplers.
- Normalization to the ideal peak allows comparison of response shapes, but does not determine actual insertion loss or the power efficiency of the two distributions.
- Gaussian weights are defined mathematically; their physical implementation in the device geometry has not yet been verified.
- Phase errors are not linked to specific variations in waveguide width, height, or length, and are assumed to be wavelength independent.
- The passing fraction is a simulation result, not an established manufacturing yield.
- Tolerances are evaluated on a discrete grid with a 0.0025 nm step for 1000 realizations; 95% is the observed passing fraction, not a statistical guarantee.
